# 33 - Where dead time flips the gain

**Purpose.** `31` found gain 50 wins nearly the whole trade-off curve, and said why: every sub
costs 31.5 s of dead time, and gain 200 needs subs about six times shorter for the same clipping.
That 31.5 s is what *dithering after every frame* costs, and dithering less often is a setting.
This notebook finds **the dead time below which gain 200 (or 100) beats gain 50**, at each share
of clipped stars, and turns that dead time into **"dither every k frames"**.

**What it computes.** `31`'s curve with one thing changed: `t_dead`, swept from 0 to the 31.5 s
measured. Clipping does not depend on `t_dead` - it is a property of one sub - so it is read
straight from `pareto_curve.csv`, not recomputed. Only the sub count, and so the night's SNR,
moves. For each clipping budget, the best setting at each gain is compared, and the flip is where
the ratio crosses 1. Beside it, for the same budget, how much SNR the best setting gains against
today's 31.5 s: the same arithmetic read the other way.

**Turning dead time into a cadence.** `t_dead(k) = download + settle / k` when dithering every
`k`-th frame. `17` measured `t_dead` only with dithering every frame, so the split between download
and settle is **inferred, not measured** (`sky_constants.json`, `t_dead_decomposition`). Two values
of the download are carried, because the record holds two and neither is settled: **0.68 s**, the
archive's figure on frames that skipped a dither (not ours), and **7.99 s**, the shortest gap of
our own night, which is only an upper bound. The cadence is reported under both.

**Fixed, as in `31`.** NGC 7000, the night of 2026-09-05, its length and sky. SNR in blue, faint
signal at 1% of sky. A star clips if its core reaches `linear_to_at_least` in any plane. Gains 50,
100 and 200. Sub lengths on `31`'s grid, 10-600 s. **`eta_comb = 1`**, as agreed for `31`.

**What it writes.** `results/dither_flip.csv`, the comparison at every dead time and budget, and
`results/dither_flip_constants.json`, the flips and cadences with their provenance. **Model
predictions, nothing measured.** `34` reads them back.

**What it is not for.** Not the cost of dithering less. Stacking loss was measured only with
dithering every frame and only up to 18 subs; fewer dither positions can only make rejection and
fixed pattern worse, and this notebook does not price that. It therefore gives the most favourable
case for dithering less, and the most favourable case for gain 200. Not other nights or targets. Not
the mount's limit on sub length.

**It assumes** `32_tradeoff_curve_read.ipynb` for the curve and why dead time decides it.

In [ ]:
import json
import math
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
mc = json.loads((RESULTS / "model_constants.json").read_text())
sky = json.loads((RESULTS / "sky_constants.json").read_text())
curve = pd.read_csv(RESULTS / "pareto_curve.csv")
frames = pd.read_csv(RESULTS / "sky_frames.csv", parse_dates=["t"])

G_TABLE = mc["system_gain"]["value"]
RN_TABLE = mc["read_noise_counts"]["value"]
F_SKY_B = mc["F_sky"]["value"]["B"]
DARK = mc["dark_current_bound"]["value"]
T_DEAD = mc["t_dead"]["value"]
DECOMP = sky["t_dead_decomposition"]["value"]
DOWNLOADS = {"archive": DECOMP["download_and_save_archive"],
             "bound": DECOMP["download_and_save_bound"]}
ARCHIVE_EVERY_2ND = (17.0, 20.0)      # the archive's t_dead, dithering every 2nd frame (t_dead note)
ETA = 1.0                             # as agreed for 31
F_OBJ_FRAC = 0.01                     # as in 31: faint is 1% of sky
GAINS = (50, 100, 200)
BUDGETS = (0.005, 0.01, 0.02, 0.05, 0.10)
TD_GRID = np.unique(np.append(np.round(np.arange(0.0, T_DEAD, 0.1), 1), T_DEAD))

night = frames.sort_values("t")
T_NIGHT = ((night.t.iloc[-1] - night.t.iloc[0]).total_seconds() + night.exptime.iloc[-1])
FLIP_CSV = RESULTS / "dither_flip.csv"
CONSTANTS = RESULTS / "dither_flip_constants.json"
print(f"night {T_NIGHT / 3600:.2f} h, t_dead today {T_DEAD:.2f} s, "
      f"{len(TD_GRID)} dead times 0-{T_DEAD:.2f} s, downloads {DOWNLOADS}")

## 1. One sub's SNR, and the check against `31`

The SNR of one sub depends on the gain and the sub length, not on dead time. The night's SNR is
that times `sqrt(N)`, and only `N` moves with dead time. Recomputed here with the model, and
checked against `31`'s published `snr_B` at the dead time `31` used: if they disagree, this notebook
is not drawing the same curve.

In [ ]:
def snr_one_sub(gain, t):
    read_e = M.read_noise_e(gain, RN_TABLE, G_TABLE)
    return M.snr_sub(F_OBJ_FRAC * F_SKY_B, t, f_sky=F_SKY_B, dark=DARK, read_e=read_e)


base = curve[curve.gain.isin(GAINS)][["gain", "t", "clipped_any", "snr_B"]].copy()
base["snr_sub"] = [snr_one_sub(g, t) for g, t in zip(base.gain, base.t)]
check = [ETA * math.sqrt(M.subs_in(T_NIGHT, t, T_DEAD)) * s
         for t, s in zip(base.t, base.snr_sub)]
worst = float(np.max(np.abs(np.array(check) / base.snr_B - 1)))
print(f"{len(base)} settings; largest disagreement with 31's snr_B: {worst:.1e}")
assert worst < 1e-5, "not 31's curve"

## 2. Every dead time: the best setting at each gain, per budget

For each dead time and each budget, the best setting at each gain among those clipping no more
than the budget. `ratio_200_50` above 1 means gain 200 wins at that budget. `snr_vs_today` is the
overall best against the overall best at 31.5 s, at the same budget: what the shorter dead time
buys whichever gain wins it.

In [ ]:
def best_at(td):
    n = np.array([M.subs_in(T_NIGHT, t, td) for t in base.t])
    return base.assign(n_subs=n, snr=ETA * np.sqrt(n) * base.snr_sub)


today = best_at(T_DEAD)
rows = []
for td in TD_GRID:
    s = best_at(td)
    for b in BUDGETS:
        ok = s[s.clipped_any <= b]
        top = {g: ok[ok.gain == g].snr.max() if (ok.gain == g).any() else np.nan for g in GAINS}
        win = ok.loc[ok.snr.idxmax()]
        rows.append({"t_dead": td, "budget": b, "gain": int(win.gain), "t": win.t,
                     "n_subs": int(win.n_subs), "ratio_100_50": top[100] / top[50],
                     "ratio_200_50": top[200] / top[50],
                     "snr_vs_today": win.snr / today[today.clipped_any <= b].snr.max()})
flip = pd.DataFrame(rows)
show = flip[flip.t_dead.isin([0.0, 2.0, 4.0, 8.0, 16.0, round(T_DEAD, 1), T_DEAD])]
print(show[show.budget.isin([0.01, 0.02, 0.05])].round(3).to_string(index=False))

## 3. The flip, per budget

The largest dead time at which the higher gain still beats gain 50. Checked to be a single
crossing: below the flip the higher gain wins at every dead time, above it never. Where the higher
gain never wins, already wins today, or has no setting under the budget at all, the flip is a null
with the reason.

Then a check on the grid: if gain 50's best setting at a budget is the grid's longest sub, gain 50
is held back by where `31`'s grid stops, and that budget's flip says nothing about the physics.

In [ ]:
ALREADY = "wins at every dead time up to today's"


def crossing(sub, col):
    wins = sub[col].to_numpy() >= 1.0
    if np.isnan(sub[col]).all():
        return None, "no setting at this gain clips this few stars on the 10-600 s grid"
    if not wins.any():
        return None, "loses at every dead time, zero included"
    if wins.all():
        return None, f"{ALREADY} {T_DEAD:.2f} s: no flip to find below it"
    last = int(np.flatnonzero(wins).max())
    assert wins[:last + 1].all(), f"{col} crosses 1 more than once"
    return float(sub.t_dead.iloc[last]), None


flips = {}
for b in BUDGETS:
    sub = flip[flip.budget == b].sort_values("t_dead")
    flips[b] = {}
    for col, key in (("ratio_200_50", "200_vs_50"), ("ratio_100_50", "100_vs_50")):
        td, why = crossing(sub, col)
        flips[b][key] = td if why is None else {"null": why}
        print(f"{100 * b:4.1f}% clipped, gain {key[:3]} over 50: "
              + (f"wins below t_dead = {td:.1f} s" if why is None else why))

T_MAX = base.t.max()
at_edge = []
for b in BUDGETS:
    ok = today[(today.gain == 50) & (today.clipped_any <= b)]
    r = ok.loc[ok.snr.idxmax()]
    if r.t >= T_MAX:
        at_edge.append(b)
        print(f"\n{100 * b:g}%: gain 50's best sub is the grid's last, {T_MAX:.0f} s, and clips only "
              f"{100 * r.clipped_any:.1f}% - it cannot spend the budget, so this flip is set by "
              f"where the grid stops, not by the physics")

## 4. From dead time to dither cadence

`t_dead(k) = download + settle / k`, with `settle = t_dead_today - download`, under each of the two
downloads. First the one check the record allows: the archive dithered every second frame and
measured 17-20 s. Then the cadence `k` that brings dead time down to each flip. If the flip sits
below the download, no cadence reaches it: saving the frame alone already costs more.

In [ ]:
for name, dl in DOWNLOADS.items():
    settle = T_DEAD - dl
    print(f"download {dl:.2f} s ({name}): settle {settle:.2f} s; every 2nd frame -> "
          f"{dl + settle / 2:.1f} s, against the archive's {ARCHIVE_EVERY_2ND[0]:.0f}-"
          f"{ARCHIVE_EVERY_2ND[1]:.0f} s")


def cadence(td_flip, dl):
    if isinstance(td_flip, dict):
        return 1 if td_flip["null"].startswith(ALREADY) else {"null": "no flip to reach"}
    if td_flip <= dl:
        return {"null": f"the flip ({td_flip:.1f} s) is below the download ({dl:.2f} s)"}
    return math.ceil((T_DEAD - dl) / (td_flip - dl))


cadences = {b: {key: {name: cadence(td, dl) for name, dl in DOWNLOADS.items()}
                for key, td in fb.items()} for b, fb in flips.items()}
print()
for b, cb in cadences.items():
    print(f"{100 * b:4.1f}%  gain 200: {cb['200_vs_50']}   gain 100: {cb['100_vs_50']}")

**How much a cadence buys, whichever gain wins.** The best setting at each budget at the dead time
a few cadences give, against today. Read from the table above, at the nearest dead time on the grid.

In [ ]:
K = (1, 2, 4, 8, 16)
cad_rows = []
for name, dl in DOWNLOADS.items():
    for k in K:
        td = dl + (T_DEAD - dl) / k
        near = flip.loc[(flip.t_dead - td).abs() == (flip.t_dead - td).abs().min(), "t_dead"].iloc[0]
        for b in (0.01, 0.02, 0.05):
            r = flip[(flip.t_dead == near) & (flip.budget == b)].iloc[0]
            cad_rows.append({"download": name, "every k": k, "t_dead": round(td, 1),
                             "budget %": 100 * b, "gain": r.gain, "t": r.t,
                             "snr_vs_today": round(r.snr_vs_today, 3)})
print(pd.DataFrame(cad_rows).to_string(index=False))

## 5. Publish

In [ ]:
measured_on = str(night.t.iloc[0].date())
flip.round(6).to_csv(FLIP_CSV, index=False)
pct = lambda b: f"{100 * b:g}"
adv = flip[flip.t_dead == 0.0].set_index("budget")
inputs = (f"model predictions, nothing measured.  31's curve (pareto_curve.csv) with t_dead swept "
          f"0-{T_DEAD:.2f} s in 0.1 s steps; clipping read from it unchanged, SNR in plane B "
          f"recomputed with model.snr_sub and model.subs_in over a {T_NIGHT / 3600:.2f} h night and "
          f"checked against 31's snr_B to {worst:.0e}.  eta_comb = {ETA}, as agreed for 31, and never "
          f"measured at any cadence but every frame: dithering less can only lose more to stacking, "
          f"so every flip here is the most favourable case for the higher gain.  Gains {GAINS}, "
          f"sub lengths 10-600 s on 31's grid; NGC 7000, 2026-09-05")
prov = {"source_frames": int(len(frames)), "measured_on": measured_on,
        "notebook": "33_dither_flip.ipynb"}
constants = {
    "t_dead_flip": {
        "value": {pct(b): v for b, v in flips.items()},
        "unit": "s: the largest t_dead at which the named gain beats gain 50, per % of stars clipped",
        "uncertainty": 0.1, **prov,
        "note": ("uncertainty is the t_dead grid step only.  Set by the end of the sub-length grid "
                 f"({T_MAX:.0f} s), not by the physics, at "
                 f"{', '.join(pct(b) + '%' for b in at_edge) or 'no budget'}: gain 50's best "
                 "sub there is the grid's last.  " + inputs)},
    "dither_cadence_flip": {
        "value": {pct(b): v for b, v in cadences.items()},
        "unit": "frames per dither needed to reach t_dead_flip, per % clipped, per download value",
        "uncertainty": None, **prov,
        "note": (f"t_dead(k) = download + (t_dead - download) / k, with t_dead {T_DEAD:.2f} s.  "
                 f"The split is t_dead_decomposition's, INFERRED and not measured: 'archive' is the "
                 f"historic corpus's {DOWNLOADS['archive']:.2f} s, not ours; 'bound' is "
                 f"{DOWNLOADS['bound']:.2f} s, the shortest gap of our night and only an upper "
                 f"bound.  Rounded up: dithering every k-th frame or less often reaches the flip, more "
                 f"often does not.  " + inputs)},
    "ratio_at_zero_dead_time": {
        "value": {pct(b): {"200_vs_50": None if math.isnan(r.ratio_200_50) else round(r.ratio_200_50, 4),
                           "100_vs_50": None if math.isnan(r.ratio_100_50) else round(r.ratio_100_50, 4)}
                  for b, r in adv.iterrows()},
        "unit": "best SNR at the named gain over best at gain 50, same budget, t_dead = 0",
        "uncertainty": None, **prov,
        "note": ("the most the higher gain can ever win by, on this night: no download, no "
                 "dithering, no stacking loss.  None where the gain has no setting under the "
                 "budget.  " + inputs)},
}
with open(CONSTANTS, "w") as fh:
    json.dump(constants, fh, indent=2)
M.Constants.load(CONSTANTS)       # the provenance gate
print("wrote", FLIP_CSV.name, f"({len(flip)} rows),", CONSTANTS.name)